# 🔍 Real-Time Fraud Detection avec PySpark

Ce notebook démontre les concepts du projet en Python/PySpark.
Exécutable gratuitement sur Google Colab!

## Technologies démontrées:
- ✅ RDD Operations
- ✅ DataFrame API
- ✅ Spark SQL
- ✅ Machine Learning (MLlib)

## 1. Installation de PySpark

In [ ]:
!pip install pyspark==3.4.1 -q
print("✅ PySpark installé!")

## 2. Initialisation de SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import random

# Créer la session Spark
spark = SparkSession.builder \
    .appName("FraudDetection") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

sc = spark.sparkContext
print(f"✅ Spark version: {spark.version}")
print(f"✅ SparkContext: {sc}")

## 3. Génération de Données de Test

In [ ]:
import time

def generate_transactions(n, fraud_rate=0.02):
    """Génère n transactions avec un taux de fraude donné"""
    locations = ["Paris", "London", "New York", "Tokyo", "Singapore", "Dubai"]
    categories = ["RETAIL", "GROCERY", "RESTAURANT", "ONLINE", "GAMBLING", "CRYPTO"]
    channels = ["ONLINE", "POS", "ATM", "MOBILE"]
    
    transactions = []
    for i in range(n):
        is_fraud = random.random() < fraud_rate
        
        if is_fraud:
            # Transaction frauduleuse: montants élevés, catégories risquées
            amount = random.uniform(2000, 15000)
            category = random.choice(["GAMBLING", "CRYPTO", "ONLINE"])
            hour = random.randint(0, 5)  # Nuit
        else:
            # Transaction normale
            amount = random.gauss(200, 150)
            amount = max(5, abs(amount))
            category = random.choice(categories[:4])
            hour = random.randint(8, 22)
        
        transactions.append((
            f"TX{i:06d}",                           # transactionId
            f"CUST{random.randint(1, 1000):04d}",   # customerId
            f"MERCH{random.randint(1, 200):03d}",   # merchantId
            round(amount, 2),                        # amount
            "EUR",                                   # currency
            random.choice(channels),                 # channel
            random.choice(locations),                # location
            int(time.time() * 1000) - random.randint(0, 86400000),  # timestamp
            random.choice(["CREDIT", "DEBIT"]),     # cardType
            random.random() < 0.1,                   # isInternational
            category,                                # merchantCategory
            round(random.uniform(1000, 10000), 2),  # previousBalance
            is_fraud                                 # isFraud
        ))
    
    return transactions

# Générer 10,000 transactions
data = generate_transactions(10000, fraud_rate=0.03)
print(f"✅ Généré {len(data)} transactions")
print(f"   Dont {sum(1 for t in data if t[-1])} fraudes ({sum(1 for t in data if t[-1])/len(data)*100:.1f}%)")

---
# 📊 PARTIE 1: RDD Operations

Démonstration des opérations RDD fondamentales

In [ ]:
# Créer un RDD à partir des données
transactions_rdd = sc.parallelize(data)

print(f"Nombre de partitions: {transactions_rdd.getNumPartitions()}")
print(f"Nombre total: {transactions_rdd.count()}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MAP - Extraire les montants
# ═══════════════════════════════════════════════════════════════════
amounts_rdd = transactions_rdd.map(lambda tx: tx[3])  # index 3 = amount

print("MAP - Premiers montants:")
print(amounts_rdd.take(5))

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FILTER - Transactions de montant élevé (> 1000€)
# ═══════════════════════════════════════════════════════════════════
high_value_rdd = transactions_rdd.filter(lambda tx: tx[3] > 1000)

print(f"FILTER - Transactions > 1000€: {high_value_rdd.count()}")
print("Exemple:", high_value_rdd.first())

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# REDUCEBYKEY - Total par client
# ═══════════════════════════════════════════════════════════════════
amount_by_customer = transactions_rdd \
    .map(lambda tx: (tx[1], tx[3])) \
    .reduceByKey(lambda a, b: a + b)

print("REDUCEBYKEY - Top 5 clients par montant total:")
top_customers = amount_by_customer.sortBy(lambda x: -x[1]).take(5)
for cust, total in top_customers:
    print(f"  {cust}: {total:,.2f}€")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# GROUPBYKEY - Transactions par localisation
# ═══════════════════════════════════════════════════════════════════
by_location = transactions_rdd \
    .map(lambda tx: (tx[6], 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .collect()

print("Distribution par localisation:")
for loc, count in sorted(by_location, key=lambda x: -x[1]):
    print(f"  {loc}: {count} transactions")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# DÉTECTION FRAUDE avec RDD
# ═══════════════════════════════════════════════════════════════════
def detect_suspicious(tx):
    """Retourne (transaction, [raisons]) si suspecte"""
    reasons = []
    amount = tx[3]
    category = tx[10]
    is_intl = tx[9]
    
    if amount > 5000:
        reasons.append(f"HIGH_AMOUNT: {amount:.2f}€")
    if category in ["GAMBLING", "CRYPTO"]:
        reasons.append(f"RISKY_CATEGORY: {category}")
    if is_intl and amount > 1000:
        reasons.append("HIGH_INTERNATIONAL")
    
    return (tx, reasons) if reasons else None

suspicious_rdd = transactions_rdd \
    .map(detect_suspicious) \
    .filter(lambda x: x is not None)

print(f"Transactions suspectes détectées: {suspicious_rdd.count()}")
print("\nExemples:")
for tx, reasons in suspicious_rdd.take(3):
    print(f"  {tx[0]}: {tx[3]:.2f}€ - {reasons}")

---
# 📊 PARTIE 2: DataFrame API

Démonstration de l'API DataFrame avec optimisation Catalyst

In [ ]:
# Définir le schéma
schema = StructType([
    StructField("transactionId", StringType(), False),
    StructField("customerId", StringType(), False),
    StructField("merchantId", StringType(), False),
    StructField("amount", DoubleType(), False),
    StructField("currency", StringType(), False),
    StructField("channel", StringType(), False),
    StructField("location", StringType(), False),
    StructField("timestamp", LongType(), False),
    StructField("cardType", StringType(), False),
    StructField("isInternational", BooleanType(), False),
    StructField("merchantCategory", StringType(), False),
    StructField("previousBalance", DoubleType(), False),
    StructField("isFraud", BooleanType(), False)
])

# Créer le DataFrame
df = spark.createDataFrame(data, schema)

print("Schema:")
df.printSchema()

print("\nAperçu des données:")
df.show(5, truncate=False)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# withColumn - Ajouter des colonnes calculées
# ═══════════════════════════════════════════════════════════════════
df_enriched = df \
    .withColumn("hour", (col("timestamp") / 3600000 % 24).cast("int")) \
    .withColumn("isHighAmount", col("amount") > 1000) \
    .withColumn("amountCategory",
        when(col("amount") < 100, "LOW")
        .when(col("amount") < 500, "MEDIUM")
        .when(col("amount") < 2000, "HIGH")
        .otherwise("VERY_HIGH")
    ) \
    .withColumn("riskScore",
        when(col("amount") > 5000, 0.4).otherwise(0.0) +
        when(col("merchantCategory").isin("GAMBLING", "CRYPTO"), 0.3).otherwise(0.0) +
        when(col("isInternational") & (col("amount") > 1000), 0.2).otherwise(0.0)
    )

print("DataFrame enrichi:")
df_enriched.select("transactionId", "amount", "hour", "amountCategory", "riskScore").show(10)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# groupBy + agg - Agrégations par client
# ═══════════════════════════════════════════════════════════════════
customer_stats = df.groupBy("customerId").agg(
    count("*").alias("txCount"),
    sum("amount").alias("totalAmount"),
    avg("amount").alias("avgAmount"),
    max("amount").alias("maxAmount"),
    sum(when(col("isFraud"), 1).otherwise(0)).alias("fraudCount")
).orderBy(desc("totalAmount"))

print("Statistiques par client (Top 10):")
customer_stats.show(10)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Window Functions - Calculs glissants
# ═══════════════════════════════════════════════════════════════════
window_spec = Window.partitionBy("customerId").orderBy("timestamp")

df_with_window = df \
    .withColumn("txNumber", row_number().over(window_spec)) \
    .withColumn("cumulativeAmount", sum("amount").over(window_spec)) \
    .withColumn("prevAmount", lag("amount", 1).over(window_spec)) \
    .withColumn("amountChange", col("amount") - col("prevAmount"))

print("Avec Window Functions:")
df_with_window \
    .filter(col("customerId") == "CUST0001") \
    .select("transactionId", "amount", "txNumber", "cumulativeAmount", "prevAmount", "amountChange") \
    .show(10)

---
# 📊 PARTIE 3: Spark SQL

Démonstration des requêtes SQL sur données distribuées

In [ ]:
# Créer une vue temporaire
df.createOrReplaceTempView("transactions")

print("✅ Vue 'transactions' créée")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Requête 1: Statistiques globales
# ═══════════════════════════════════════════════════════════════════
spark.sql("""
    SELECT
        COUNT(*) as total_transactions,
        ROUND(SUM(amount), 2) as total_volume,
        ROUND(AVG(amount), 2) as avg_amount,
        SUM(CASE WHEN isFraud THEN 1 ELSE 0 END) as fraud_count,
        ROUND(SUM(CASE WHEN isFraud THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as fraud_rate_percent
    FROM transactions
""").show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Requête 2: Analyse par catégorie de marchand
# ═══════════════════════════════════════════════════════════════════
spark.sql("""
    SELECT
        merchantCategory,
        COUNT(*) as tx_count,
        ROUND(SUM(amount), 2) as total_volume,
        ROUND(AVG(amount), 2) as avg_amount,
        SUM(CASE WHEN isFraud THEN 1 ELSE 0 END) as fraud_count,
        ROUND(SUM(CASE WHEN isFraud THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as fraud_rate
    FROM transactions
    GROUP BY merchantCategory
    ORDER BY fraud_rate DESC
""").show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Requête 3: Transactions à haut risque avec CASE WHEN
# ═══════════════════════════════════════════════════════════════════
spark.sql("""
    SELECT
        transactionId,
        customerId,
        amount,
        merchantCategory,
        CASE
            WHEN amount > 5000 THEN 'VERY_HIGH_AMOUNT'
            WHEN merchantCategory IN ('GAMBLING', 'CRYPTO') THEN 'RISKY_MERCHANT'
            WHEN isInternational AND amount > 1000 THEN 'HIGH_INTERNATIONAL'
            ELSE 'NORMAL'
        END as risk_reason,
        CASE
            WHEN amount > 5000 OR merchantCategory IN ('GAMBLING', 'CRYPTO') THEN 'HIGH'
            WHEN amount > 1000 THEN 'MEDIUM'
            ELSE 'LOW'
        END as risk_level,
        isFraud
    FROM transactions
    WHERE amount > 1000 OR merchantCategory IN ('GAMBLING', 'CRYPTO')
    ORDER BY amount DESC
    LIMIT 15
""").show(truncate=False)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Requête 4: CTE (WITH) pour analyse complexe
# ═══════════════════════════════════════════════════════════════════
spark.sql("""
    WITH customer_stats AS (
        SELECT
            customerId,
            AVG(amount) as avg_amount,
            STDDEV(amount) as std_amount,
            COUNT(*) as tx_count
        FROM transactions
        GROUP BY customerId
        HAVING COUNT(*) >= 3
    )
    SELECT
        t.transactionId,
        t.customerId,
        t.amount,
        ROUND(cs.avg_amount, 2) as customer_avg,
        ROUND((t.amount - cs.avg_amount) / cs.std_amount, 2) as z_score,
        CASE
            WHEN ABS((t.amount - cs.avg_amount) / cs.std_amount) > 3 THEN 'EXTREME_OUTLIER'
            WHEN ABS((t.amount - cs.avg_amount) / cs.std_amount) > 2 THEN 'OUTLIER'
            ELSE 'NORMAL'
        END as anomaly_status
    FROM transactions t
    JOIN customer_stats cs ON t.customerId = cs.customerId
    WHERE ABS((t.amount - cs.avg_amount) / NULLIF(cs.std_amount, 0)) > 2
    ORDER BY z_score DESC
    LIMIT 10
""").show()

---
# 🤖 PARTIE 4: Machine Learning (MLlib)

Entraînement d'un modèle de détection de fraude

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline

print("✅ MLlib importé")

In [ ]:
# Préparer les données pour ML
ml_data = df_enriched.select(
    "amount", "previousBalance", "hour", "riskScore",
    "channel", "cardType", "merchantCategory", "isInternational",
    col("isFraud").cast("double").alias("label")
)

# Split train/test
train_data, test_data = ml_data.randomSplit([0.8, 0.2], seed=42)

print(f"Training set: {train_data.count()}")
print(f"Test set: {test_data.count()}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Créer le Pipeline ML
# ═══════════════════════════════════════════════════════════════════

# 1. Indexer les colonnes catégorielles
channel_indexer = StringIndexer(inputCol="channel", outputCol="channelIndex", handleInvalid="keep")
card_indexer = StringIndexer(inputCol="cardType", outputCol="cardTypeIndex", handleInvalid="keep")
merchant_indexer = StringIndexer(inputCol="merchantCategory", outputCol="merchantIndex", handleInvalid="keep")

# 2. Assembler les features
assembler = VectorAssembler(
    inputCols=["amount", "previousBalance", "hour", "riskScore", 
               "channelIndex", "cardTypeIndex", "merchantIndex"],
    outputCol="features",
    handleInvalid="skip"
)

# 3. Standardiser
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")

# 4. Random Forest Classifier
rf = RandomForestClassifier(
    featuresCol="scaledFeatures",
    labelCol="label",
    numTrees=50,
    maxDepth=10,
    seed=42
)

# Pipeline complet
pipeline = Pipeline(stages=[
    channel_indexer, card_indexer, merchant_indexer,
    assembler, scaler, rf
])

print("✅ Pipeline ML créé")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Entraîner le modèle
# ═══════════════════════════════════════════════════════════════════
print("Entraînement du modèle...")
model = pipeline.fit(train_data)
print("✅ Modèle entraîné!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Évaluer le modèle
# ═══════════════════════════════════════════════════════════════════
predictions = model.transform(test_data)

# Évaluateur
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction")

auc = evaluator.evaluate(predictions, {evaluator.metricName: "areaUnderROC"})
aupr = evaluator.evaluate(predictions, {evaluator.metricName: "areaUnderPR"})

# Calculer précision, recall
tp = predictions.filter((col("prediction") == 1) & (col("label") == 1)).count()
fp = predictions.filter((col("prediction") == 1) & (col("label") == 0)).count()
fn = predictions.filter((col("prediction") == 0) & (col("label") == 1)).count()
tn = predictions.filter((col("prediction") == 0) & (col("label") == 0)).count()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("="*50)
print("       RÉSULTATS DU MODÈLE")
print("="*50)
print(f"AUC-ROC:    {auc:.4f}")
print(f"AUC-PR:     {aupr:.4f}")
print(f"Precision:  {precision:.4f}")
print(f"Recall:     {recall:.4f}")
print(f"F1-Score:   {f1:.4f}")
print("-"*50)
print("Matrice de confusion:")
print(f"  TP: {tp:5d} | FP: {fp:5d}")
print(f"  FN: {fn:5d} | TN: {tn:5d}")
print("="*50)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Exemples de prédictions
# ═══════════════════════════════════════════════════════════════════
print("Exemples de prédictions:")
predictions.select(
    "amount", "merchantCategory", "riskScore",
    "label", "prediction", "probability"
).show(15, truncate=False)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Importance des features
# ═══════════════════════════════════════════════════════════════════
rf_model = model.stages[-1]
feature_names = ["amount", "previousBalance", "hour", "riskScore", 
                 "channel", "cardType", "merchantCategory"]

print("\nImportance des features:")
for name, importance in zip(feature_names, rf_model.featureImportances):
    print(f"  {name:20s}: {importance:.4f} {'█' * int(importance * 50)}")

---
# 🎉 Résumé

Ce notebook a démontré:

1. **RDD Operations**: map, filter, reduceByKey, groupByKey
2. **DataFrame API**: withColumn, groupBy, agg, Window functions
3. **Spark SQL**: Requêtes SQL, CTE, CASE WHEN, agrégations
4. **MLlib**: Pipeline ML, Random Forest, évaluation

Tous les concepts du projet Real-Time Fraud Detection!

In [ ]:
# Arrêter Spark
spark.stop()
print("✅ Session Spark terminée")